# 09 — Grupimi dhe zbulimi i strukturës

        **Pyetja udhëheqëse:** Si gjejmë regjime pa etiketa të njohura?

        Ky notebook është ndërtuar si një mësim i plotë kompjuterik. Rruga kryesore ndjek:
        **problem konkret → model matematikor → zbatim nga parimet e para → bibliotekë profesionale → krahasim → interpretim**.

        ## Objektivat
        - Të implementohet k-means nga zero.
- Të kuptohet funksioni objektiv dhe inicializimi.
- Të përdoren elbow dhe silhouette.
- Të krahasohet me grupimin hierarkik.

        > **Rruga bazë për orën mësimore (45–60 min)** shënohet me seksionet **CORE**.  
        > Seksionet **EXTENDED** dhe **PLAYGROUND** janë për thellim dhe punë të pavarur.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import time

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("09 — Grupimi dhe zbulimi i strukturës")
print("Seed:", RANDOM_STATE)

# 1. Regjime atmosferike pa etiketa — CORE

In [ ]:
from sklearn.datasets import make_blobs
X,_=make_blobs(n_samples=520,centers=[[-2,-1],[0,2.3],[2.6,-0.4]],
               cluster_std=[0.6,0.75,0.65],random_state=RANDOM_STATE)
fig,ax=plt.subplots()
ax.scatter(X[:,0],X[:,1],s=14)
ax.set_xlabel("anomalia e temperaturës"); ax.set_ylabel("anomalia e presionit")
ax.set_title("Të dhëna pa etiketa")
plt.show()

# 2. Funksioni objektiv i k-means — CORE

k-means minimizon
$$
J=\sum_{k=1}^{K}\sum_{x_i\in C_k}\|x_i-\mu_k\|^2.
$$
Algoritmi alternon ndërmjet caktimit te qendra më e afërt dhe rillogaritjes së qendrave.

# 3. Iteracionet manuale — CORE

In [ ]:
K=3
centers=X[rng.choice(len(X),K,replace=False)].copy()
history=[centers.copy()]
labels_history=[]
for _ in range(8):
    d2=((X[:,None,:]-centers[None,:,:])**2).sum(axis=2)
    labels=d2.argmin(axis=1)
    labels_history.append(labels.copy())
    new=np.array([X[labels==k].mean(axis=0) for k in range(K)])
    history.append(new.copy())
    if np.linalg.norm(new-centers)<1e-6: break
    centers=new

for step in [0,1,min(2,len(labels_history)-1),len(labels_history)-1]:
    fig,ax=plt.subplots()
    lab=labels_history[min(step,len(labels_history)-1)]
    ax.scatter(X[:,0],X[:,1],c=lab,s=10)
    c=history[step]
    ax.scatter(c[:,0],c[:,1],s=130,marker="x")
    ax.set_title(f"k-means: iteracioni {step}")
    plt.show()

### Zbatimi kanonik i k-means në `src/`

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd()))
from src.clustering.kmeans import KMeansFromScratch
km_src = KMeansFromScratch(n_clusters=3, seed=RANDOM_STATE).fit(X)
print("Inertia nga src:", round(km_src.inertia_, 3), "iteracione:", km_src.n_iter_)

# 4. Elbow dhe silhouette — CORE

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
ks=range(2,8); inertia=[]; sil=[]
for k in ks:
    m=KMeans(k,random_state=RANDOM_STATE,n_init=10).fit(X)
    inertia.append(m.inertia_)
    sil.append(silhouette_score(X,m.labels_))

fig,ax=plt.subplots()
ax.plot(list(ks),inertia,marker="o")
ax.set_xlabel("k"); ax.set_ylabel("Inertia")
ax.set_title("Metoda e bërrylit")
plt.show()

fig,ax=plt.subplots()
ax.plot(list(ks),sil,marker="o")
ax.set_xlabel("k"); ax.set_ylabel("Silhouette")
ax.set_title("Silhouette score")
plt.show()

# 5. Grupimi hierarkik — CORE

In [ ]:
from scipy.cluster.hierarchy import linkage,dendrogram
Z=linkage(X[:80],method="ward")
fig,ax=plt.subplots(figsize=(10,5))
dendrogram(Z,no_labels=True,ax=ax)
ax.set_ylabel("Distanca e bashkimit")
ax.set_title("Dendrogram")
plt.show()

---
## Deri këtu: materiali bazë për orën mësimore

Në një seancë rreth 60 minuta, zakonisht ndalemi këtu. Seksionet që vijojnë janë për thellim,
ushtrime të pavarura dhe eksperimentim.

# 6. Kur k-means dështon — EXTENDED

In [ ]:
from sklearn.datasets import make_moons
Xm,_=make_moons(n_samples=450,noise=0.07,random_state=RANDOM_STATE)
km=KMeans(2,random_state=RANDOM_STATE,n_init=10).fit(Xm)
fig,ax=plt.subplots()
ax.scatter(Xm[:,0],Xm[:,1],c=km.labels_,s=12)
ax.set_title("k-means imponon grupe afërsisht sferike")
plt.show()

**Interpretim.** k-means optimizon një funksion të caktuar gjeometrik; ai nuk 'zbulon' domosdoshmërisht grupet që ne konsiderojmë fizikisht të kuptimshme.

# 7. Aplikim jo-fizik: segmentim klientësh — EXTENDED

Në marketing, të njëjtat mjete përdoren për segmentim, por interpretimi i grupeve duhet të bëhet me kujdes: grupet algoritmike nuk janë kategori natyrore të garantuara.

# Ushtrime — PLAYGROUND

        ## A. Ushtrime të shkurtra
        1. Përsëritni k-means me 20 seed dhe analizoni ndryshueshmërinë.
2. Krahasoni k=2,3,4 me silhouette.
3. Ndryshoni `linkage` në dendrogram.
4. Shpjegoni pse scaling ndryshon grupimin.

        ## B. Ushtrim i plotë
        Gjeneroni tri faza fizike me forma jo-sferike dhe krahasoni k-means me AgglomerativeClustering.

        ## C. Playground
        Ndryshoni k, seed, zhurmën dhe standardizimin.

In [ ]:
# TODO / PLAYGROUND
# Ndryshoni parametrat, shtoni grafikë ose krahasoni modele të tjera.